# 📈 Stock Analyzer Core - 日本株定量スクリーニング & クオンツ分析基盤

東証全上場約3,900社を対象に、財務諸表（EDINET）と日足市場データを統合し、3層構造（事前足切り・クオンツ配点・ゲートキーパー）によって優良銘柄を高速スクリーニングする実行ノートブックです。

---

### ⚠️ 免責事項（Disclaimer）および利用規約の遵守

> **【投資判断に関する免責事項】**  
> 本ツールおよび解説コンテンツは、公開財務データおよび市場データの自動処理・スクリーニング技術の学習・研究を目的としたものであり、特定の有価証券の売買推奨や投資助言を提供するものではありません。算出されるスコアおよび判定（Verdict）は数式モデルに基づく機械的計算結果であり、将来の運用成果を保証するものではありません。実際の投資判断および売買は、必ずご自身の責任において行ってください。また、本ツールの利用により生じたいかなる損害についても、著者は一切の責任を負いかねます。

> **【データ出典および利用境界】**  
> - **EDINET**: 出典 金融庁 EDINET 閲覧（提出）サイト。本ツールは政府のオープンデータ基本指針に基づき、利用者個人のAPIキーにて通信を行います。
> - **yfinance**: 本ツールは個人学習・研究を目的として yfinance を使用しています。Yahoo Finance の利用規約上の制限やアクセス制限リスク（429 Too Many Requests 等）が存在するため、安定した本番運用や商用利用を行う場合は、日本取引所グループ公式の [J-Quants API](https://jquants.com/) 等の正規データプロバイダの利用を強く推奨します。
---

### 📜 ライセンス & 著作権表示（License & Copyright）
- **Copyright**: (c) 2026 akkyey (`akkyey/stock-analyzer-core`)
- **License**: 本ノートブックおよび付随するプログラムコードは [MIT License](https://opensource.org/licenses/MIT) の下で公開・提供されています。
- 商用利用・私的利用・改変・再配布が許諾されていますが、著作権表示および許諾表示の保持が必要です。


## 【Step 0】Google Drive マウント & キャッシュ初期化（Pull Phase）
Google Drive をマウントし、前回のキャッシュデータベースを作業用ローカル超高速SSDへロードします。

In [ ]:
# @title 【Step 0】Google Drive マウント & キャッシュ初期化（Pull Phase） { run: "auto" }
# @markdown ### 📁 Google Drive 上の保存先フォルダ設定
# @markdown マイドライブ直下のフォルダ名、または `/content/drive/...` のフルパスを指定できます（デフォルト: `StockAnalyzer`）
drive_folder_name = "StockAnalyzer"  # @param {type:"string"}

# @markdown ### 🔄 データベース初期化スイッチ
# @markdown キャッシュ破損時や、ゼロから全件再スキャン（初期構築）をやり直したい場合はチェックを入れてください
reset_database = False  # @param {type:"boolean"}

import os
import shutil
from pathlib import Path

# 1. Google Drive のマウント (例外ハンドリング & 未マウント時フォールバック)
is_drive_mounted = False
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    is_drive_mounted = os.path.exists('/content/drive/MyDrive')
except ImportError:
    print("ℹ️ ローカル環境で実行中です。Google Drive マウントをスキップします。")
except Exception as e:
    print(f"⚠️ Google Drive のマウントに失敗しました ({e})。")
    print("   Google Drive なしでもローカル超高速SSD（/content/working）のみで全スクリーニングを実行可能です。")

# 2. パス定数の設定 (ユーザー指定フォルダの安全な解決)
raw_drive_name = globals().get("drive_folder_name", "StockAnalyzer").strip() or "StockAnalyzer"
if raw_drive_name.startswith("/"):
    DRIVE_DIR = Path(raw_drive_name)
else:
    DRIVE_DIR = Path("/content/drive/MyDrive") / raw_drive_name

os.environ["STOCK_ANALYZER_DRIVE_DIR"] = str(DRIVE_DIR)
WORKING_DIR = Path("/content/working")

# 永続層ディレクトリの安全な作成
if is_drive_mounted:
    try:
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        (DRIVE_DIR / "cache").mkdir(exist_ok=True)
        (DRIVE_DIR / "output").mkdir(exist_ok=True)
        (DRIVE_DIR / "config").mkdir(exist_ok=True)
        print(f"✅ Google Drive 永続層の接続完了: {DRIVE_DIR}")
    except Exception as e:
        print(f"⚠️ Google Drive フォルダの作成中に警告 ({e})。ローカル作業層のみで続行します。")
        is_drive_mounted = False
else:
    print("ℹ️ Google Drive 未マウントのため、本セッションはローカル作業層のみで実行されます。")
    print("   （※セッション終了時に成果物がクリアされるため、Step 5 でのCSVダウンロードを推奨します）")

# 作業層ディレクトリの作成 (必須ローカルSSD)
WORKING_DIR.mkdir(parents=True, exist_ok=True)
(WORKING_DIR / "cache").mkdir(exist_ok=True)
(WORKING_DIR / "output").mkdir(exist_ok=True)
(WORKING_DIR / "tmp").mkdir(exist_ok=True)

print(f"✅ 作業層 (内蔵ローカルSSD) 準備完了: {WORKING_DIR}")


## 【Step 1】環境セットアップ（コード取得 ＆ 依存ライブラリ導入）
公開 GitHub リポジトリから安定版コードを取得し、実行に必要な最小依存関係をインストールします。

In [ ]:
import os
import shutil
import subprocess
import sys

# 作業ディレクトリを /content に戻してから操作 (再実行時の作業ディレクトリ消失エラー防止)
os.chdir("/content")

REPO_URL = "https://github.com/akkyey/stock-analyzer-core.git"
TARGET_DIR = "/content/stock-analyzer-core"
TARGET_BRANCH = "release/v1.3.0"  # 【検証中】Colab 実機確認用。リリース時に検証済みタグ (v1.3.0) へ戻す

if os.path.exists(TARGET_DIR):
    shutil.rmtree(TARGET_DIR, ignore_errors=True)

# Git クローンの耐障害性向上 (2回リトライ & main ブランチ自動フォールバック)
clone_success = False
current_branch = TARGET_BRANCH

for attempt in range(1, 3):
    try:
        print(f"📥 リポジトリを取得中 (attempt {attempt}/2, branch/tag: {current_branch})...")
        res = subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", current_branch, REPO_URL, TARGET_DIR],
            check=True,
            capture_output=True,
            text=True,
        )
        clone_success = True
        break
    except subprocess.CalledProcessError as e:
        err_msg = e.stderr.strip() if e.stderr else str(e)
        print(f"⚠️ Git クローン警告 (attempt {attempt}): {err_msg}")
        if attempt == 1:
            print("   main ブランチへフォールバックして再試行します...")
            current_branch = "main"
        if os.path.exists(TARGET_DIR):
            shutil.rmtree(TARGET_DIR, ignore_errors=True)

if clone_success and current_branch != TARGET_BRANCH:
    print(f"⚠️ 安定版 ({TARGET_BRANCH}) を取得できなかったため、最新の開発版で続行します。")
    print("   通常は問題なく動作しますが、表示や挙動が説明と異なる場合は、時間をおいて Step 1 から再実行してください。")
if not clone_success:
    raise RuntimeError("❌ GitHub からのリポジトリ取得に失敗しました。インターネット接続または GitHub の稼働状況を確認してセルを再実行してください。")

try:
    os.chdir(TARGET_DIR)
    subprocess.run(["pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
except subprocess.CalledProcessError as e:
    raise RuntimeError(f"❌ 依存ライブラリのインストールに失敗しました: {e}。セルを再実行してください。") from None

# 既存のキャッシュモジュールを安全に破棄して最新版を確実にロード
for mod in list(sys.modules.keys()):
    if mod.startswith("src"):
        del sys.modules[mod]

# Stage-and-Sync 管理モジュールによる DB ロード (3段構えの復元検証)
from src.utils.colab_sync import ColabSyncManager

# 0. 初期化スイッチの処理 (Drive およびローカルキャッシュのクリーン消去)
if globals().get("reset_database", False):
    print("🧹 【初期化スイッチ検知】既存のデータベースとキャッシュをクリーン消去します...")
    print(f"   消去対象: {DRIVE_DIR} (cache/ のデータベース) と、この実行環境の作業領域")
    ColabSyncManager.reset_cache(DRIVE_DIR, WORKING_DIR)
    print("✨ キャッシュの消去が完了しました。初回実行モードとして初期構築を行います。")
    print("⚠️ 【重要】完了後は Step 0 の「reset_database」のチェックを外してください。")
    print("   ONのままだと、次回の実行でもデータベースが消去されてしまいます。")

# Drive 再マウントガード (Step 3 完了時の同期確定でアンマウントされた後に Step 1 を再実行した場合など)
if not os.path.exists("/content/drive/MyDrive"):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass

# DB の配置 (Drive のメイン → .bak → 作業層の既存 DB → 新規 の順)。
# 既存 DB を使う場合は整合性を検証し、破損・履歴不足時はガイド付きで安全に中断する
db_path = ColabSyncManager.pull_database(DRIVE_DIR, WORKING_DIR, validate_integrity=True)

# 初回判定は「実際に使う DB」の有無で決める (Drive が見えないだけで初回扱いにしない)
is_first_run = not ColabSyncManager.is_duckdb_healthy(db_path)
# DB の出どころ: Drive 上の保存済み DB か、この実行環境の作業領域に残っていた DB か
drive_has_db = not ColabSyncManager.is_first_run(DRIVE_DIR)
print(f"✅ 実行用 DB 準備完了: {db_path}")

if is_first_run:
    print()
    print("━" * 60)
    print("ℹ️ 【初回実行モードを検知しました】")
    print("   Google Drive に前回のキャッシュが存在しないため、初回全件スキャンを実行します。")
    print("   Step 3 の実行には【約 18〜20 分】かかります。ブラウザを閉じずにお待ちください ☕")
    print("   （※全3,890銘柄の過去1年分日足＋EDINET開示データを安全に一括初期構築します）")
    print("━" * 60)
elif drive_has_db:
    print()
    print("━" * 60)
    print("⚡ 【差分更新モード（2回目以降）を検知しました】")
    print("   Google Drive から前回のキャッシュDBをロードしました。")
    print("   既存DBをもとに全銘柄の直近差分株価を高速取得します。")
    print("   所要時間は【約 5〜8 分】となります。ブラウザを閉じずにお待ちください ☕")
    print("━" * 60)
else:
    print()
    print("━" * 60)
    print("⚠️ 【Drive に保存済みのDBがありません】")
    print("   この実行環境に残っていた前回のDBを、そのまま引き続き使います（差分更新モード）。")
    print("   保存先フォルダを変更した場合や、ゼロからやり直したい場合は、Step 0 の")
    print("   「reset_database」を ON にして Step 0 から実行し直してください。")
    print("━" * 60)


## 【Step 2】EDINET API キー設定 & 疎通確認
EDINET から有報・四半報を取得するための API キーを設定します。
Colab の「シークレット（左サイドバーの鍵アイコン 🔑）」に `EDINET_API_KEY` を登録するか、以下のセルで対話入力してください。
※【注】EDINET アカウント発行時には電話番号（SMS）認証が必要です。

In [ ]:
import getpass
import os
import re
from datetime import datetime, timedelta

import requests

# 1. API キーの取得
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get('EDINET_API_KEY')
except Exception:
    pass

if not api_key:
    api_key = os.environ.get('EDINET_API_KEY')

if not api_key:
    api_key = getpass.getpass('🔑 EDINET APIキーを入力してください (画面には非表示): ')

os.environ['EDINET_API_KEY'] = api_key

# 2. 疎通テスト (ヘッダー認証 & キー伏字化 & 認証失敗時の即時中断)
test_date = (datetime.now() - timedelta(days=2)).strftime('%Y-%m-%d')
test_url = 'https://api.edinet-fsa.go.jp/api/v2/documents.json'
headers = {
    'Ocp-Apim-Subscription-Key': api_key,
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36',
}
params = {
    'date': test_date,
    'type': 2,
    'Subscription-Key': api_key,  # EDINET API v2 公式仕様準拠 (クエリパラメータ)
}

try:
    resp = requests.get(test_url, headers=headers, params=params, timeout=10)
    if resp.status_code in (401, 403):
        raise RuntimeError(f'❌ EDINET APIキーの認証に失敗しました (HTTP {resp.status_code})。正しいキーを入力してください。')

    # テキストレベルでの認証エラー検知 (JSONパース前)
    text_lower = resp.text.lower()
    if 'access denied due to invalid subscription key' in text_lower or '"statuscode": 401' in text_lower or '"statuscode": 403' in text_lower:
        raise RuntimeError('❌ EDINET APIキーの認証に失敗しました (レスポンスに認証拒否が含まれています)。正しいキーを入力してください。')

    resp.raise_for_status()
    data = resp.json()

    # 1. EDINET API v2 の実応答: HTTP 200 かつ トップレベル StatusCode / statusCode が 401 / 403
    top_code = str(data.get('StatusCode') or data.get('statusCode') or '')
    if top_code in ('401', '403'):
        err_msg = data.get('message', 'Invalid subscription key')
        raise RuntimeError(f'❌ EDINET APIキーの認証に失敗しました (StatusCode: {top_code} - {err_msg})。キーを確認してください。')

    # 2. metadata.status 判定
    meta_status = str(data.get('metadata', {}).get('status', ''))
    if meta_status in ('401', '403'):
        raise RuntimeError(f'❌ EDINET APIキーの認証に失敗しました (metadata status: {meta_status})。キーを確認してください。')

    # 3. message 内の認証拒否キーワード
    msg_lower = str(data.get('message', '')).lower()
    if 'access denied' in msg_lower or 'invalid subscription key' in msg_lower:
        raise RuntimeError(f'❌ EDINET APIキーの認証に失敗しました ({data.get("message")})。キーを確認してください。')

    print('✅ EDINET API 疎通確認成功！ 認証は有効です。')
except Exception as e:
    masked_err = re.sub(r'(Subscription-Key=)[^&]+', r'\1***', str(e))
    print(f'❌ エラー: EDINET API疎通に失敗しました: {masked_err}')
    raise RuntimeError('EDINET APIキーの認証または接続に失敗したため、後続の処理を中断します。APIキーとネットワークを確認して再実行してください。') from None


## 【Step 2.5】⚙️ スクリーニング設定のカスタマイズ
ご自身の投資スタイルに合わせて、足切り基準や配点プリセットを調整できます。
※【変更は任意です】標準の黄金比（balanced）でよろしければ、数値を変更せずそのまま再生ボタン（▶）を1回押して通過してください。

In [ ]:
# fmt: off
# @title ⚙️ スクリーニング設定のカスタマイズ { run: "auto" }

# @markdown ### 1. 外部設定ファイル (Drive 上の JSON) の利用
use_custom_json = False  # @param {type:"boolean"} # True の場合、Drive 上の config/custom_config.json を最優先

# @markdown ### 2. 投資スタイル (配点プリセット)
strategy_preset = "balanced"  # @param ["balanced", "dividend_focus", "deep_value", "growth_quality"]

# @markdown ### 3. 第1層 足切り条件 (Pre-Filter)
min_trading_value_million = 30  # @param {type:"slider", min:5, max:100, step:5} # 20日平均売買代金 (百万円)
min_price = 50.0  # @param {type:"number"} # 最低株価 (円)
target_market_ui = "全市場"  # @param ["全市場", "プライムのみ", "スタンダードのみ", "グロースのみ"]

# @markdown ### 4. 判定表記モード (Verdict Mode)
verdict_mode_ui = "客観グレード (Grade S/A/B/C: 推奨)"  # @param ["客観グレード (Grade S/A/B/C: 推奨)", "Qiita互換 (STRONG_BUY/BUY/WATCH/PASS)"]

# fmt: on

import json
from pathlib import Path

MARKET_MAP = {
    "全市場": None,
    "プライムのみ": ["Prime"],
    "スタンダードのみ": ["Standard"],
    "グロースのみ": ["Growth"],
}

# 設定の解決
if use_custom_json:
    json_path = DRIVE_DIR / "config" / "custom_config.json"
    if json_path.exists():
        with open(json_path, encoding="utf-8") as f:
            custom_config = json.load(f)
        print(f"✅ Drive 上のカスタム設定を適用しました: {json_path}")
    else:
        print(f"⚠️ {json_path} が見つからないため、Colab 画面の入力値を使用します。")
        use_custom_json = False

if not use_custom_json:
    verdict_mode = "grade" if "客観グレード" in verdict_mode_ui else "legacy"
    custom_config = {
        "hard_filters": {
            "min_trading_value": float(min_trading_value_million * 1_000_000),
            "min_price": float(min_price),
            "target_markets": MARKET_MAP.get(target_market_ui, None),
        },
        "strategy_preset": strategy_preset,
        "scoring_multipliers": {},
        "verdict_mode": verdict_mode,
    }
    print(f"✅ Colab フォームの設定を適用しました: 戦略='{strategy_preset}', 判定表記='{verdict_mode}', 売買代金>={min_trading_value_million}百万円, 最低株価>={min_price}円, 市場='{target_market_ui}'")


## 【Step 3】パイプライン実行 & Drive アトミック同期
作業用超高速SSD上で全銘柄の取得・集計・スクリーニングを実行し、完了後に最新DBと生成レポートを Google Drive へ安全にアトミック同期します。
⏳ **所要時間の目安**: 初回実行時は **約 18〜20 分（全3,890銘柄の1年分データ初期構築）**、2回目以降は **約 5〜8 分（差分更新）** です（Colab 上で安全に連続取得するため、ブラウザを閉じずにお待ちください）。


In [ ]:
import logging
import os
import time
from pathlib import Path

# 外部ライブラリの非致命的ノイズログ（404/Missing）を抑制して出力をクリーンに保つ
logging.getLogger("yfinance").setLevel(logging.CRITICAL)
logging.getLogger("urllib3").setLevel(logging.CRITICAL)


# 1. 再マウントガード (アンマウント状態であれば自動再マウント)
if not os.path.exists("/content/drive/MyDrive"):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        pass

# 2. 環境変数の設定 (必ず作業用SSDを指す)
os.environ["STOCK_ANALYZER_BASE_DIR"] = "/content/working"

# 3. パイプライン実行
from src.orchestration.context import OrchestratorContext
from src.orchestration.pipeline import OrchestrationPipeline
from src.utils.colab_sync import ColabSyncManager

# Step 1 で取得したコードが古いまま (Step 1 の未実行など) の場合は、原因と対処を案内して止める
if not hasattr(ColabSyncManager, "enforce_integrity"):
    raise RuntimeError(
        "❌ 実行中のプログラムが古いままです。「ランタイム」→「セッションを再起動」を実行し、"
        "Step 0 から順にやり直してください。"
    )
DRIVE_DIR = ColabSyncManager.resolve_drive_dir(globals().get("DRIVE_DIR"))

# 初回実行判定 (Step 1 の結果を引き継ぎ。未実行なら Drive の状態から判定)
is_first = globals().get("is_first_run")
if is_first is None:
    is_first = ColabSyncManager.is_first_run(DRIVE_DIR)

# Step 3 実行前 DB 整合性検証ガード (破損・履歴不足時のフェイルセーフ)
if not is_first:
    check_db = Path("/content/working/cache/stock_analyzer.duckdb")
    if not check_db.exists():
        check_db = Path("/content/working/stock_analyzer.duckdb")
    if check_db.exists():
        ColabSyncManager.enforce_integrity(check_db)

if is_first:
    print("⏳ 【初回実行モード】全3,890銘柄の市場データを初期スキャン中...")
    print("   所要時間は【約 18〜20 分】です（安全のため順次取得）。ブラウザを閉じずに少々お待ちください ☕")
else:
    print("⚡ 【差分更新モード】既存DBをもとに直近の市場データ差分を取得中（所要時間: 約 5〜8 分）...")
print("🚀 パイプライン実行を開始します...")
start_time = time.time()

context = OrchestratorContext(debug_mode=False)
context.config["is_first_run"] = is_first
# Colab 専用の取得プロファイル (429 に備えたリトライ拡大。事前インターバルは実測により標準と同値)
# (開発者の検証用: 事前に fetch_profile = "standard" を定義すると切り替え可能。通常は何もしない)
context.config["fetch_profile"] = globals().get("fetch_profile", "colab")
# Colab では Drive マウント経由で同期するため、サービスアカウント / スプレッドシート連携は使わない
context.config.setdefault("gdrive", {})["enabled"] = False
# カスタム設定の注入 (Step 2.5 未実行時もデフォルトで安全動作)
cfg = globals().get("custom_config", {})
if cfg:
    if hasattr(context, 'update_config'):
        context.update_config(cfg)
    else:
        context.config.update(cfg)

pipeline = OrchestrationPipeline(context)

# 調査用サマリ (トラブル時にサポートへ添付できる。APIキー等は含まれません)
from src.utils.diagnostics import (
    DIAGNOSTICS_FILENAME,
    SOURCE_COMPLETED,
    SOURCE_FAILED,
    stamp_run,
    write_run_summary,
)

stamp_run(context)
diag_path = Path("/content/working/output") / DIAGNOSTICS_FILENAME
diag_extra = {
    "取得コード指定(TARGET_BRANCH)": globals().get("TARGET_BRANCH", "?"),
    "初回実行判定(Step 3 開始時)": is_first,
    "reset_database": globals().get("reset_database", False),
    "Drive保存先": DRIVE_DIR,
}
try:
    final_df = pipeline.run()
except Exception as e:
    write_run_summary(
        diag_path, context=context, working_dir=Path("/content/working"), drive_dir=DRIVE_DIR,
        elapsed_sec=time.time() - start_time, error=e, extra=diag_extra, source=SOURCE_FAILED,
    )
    print("❌ 処理が途中で中断しました。お手数ですが、一番下の「困ったときは」のセルを実行し、")
    print("   表示された内容 (またはダウンロードされたファイル) を添えてご連絡ください。")
    raise

elapsed = time.time() - start_time
# 正常完了: 以降の再実行 (Step 3 単独再実行を含む) は差分モードとして扱う
is_first_run = False
missing_codes = sorted(set(context.config.get("fetch_missing_codes", [])))
if missing_codes:
    print(f"ℹ️ 市場データ取得対象外 / データなし: {len(missing_codes)} 件。")
    print("   上場廃止や、株価データが提供されない銘柄 (PRO Market 等) が含まれます (異常ではありません)。連続して取得できない銘柄は、30日間スキップされます。")
    print("   それ以外の取得漏れは、次回実行時に自動的に1年分で取り直されます。")
print(f"🎉 全処理が完了しました！ 所要時間: {elapsed:.2f} 秒 ({elapsed / 60:.1f} 分)")
print(f"   （うち市場データ取得: {context.perf_stats.get('fetch_sec', 0):.0f} 秒 / 取得プロファイル: {context.config['fetch_profile']}）")

# 4. Google Drive へのアトミック Push & クラウド同期保証 (flush_and_unmount)
print("📦 Google Drive へ最新成果物を同期中...")
sync_ok = ColabSyncManager.push_artifacts(WORKING_DIR, DRIVE_DIR, flush_unmount=True)
if sync_ok:
    print(f"✨ Google Drive への同期が完了しました ({DRIVE_DIR})")
else:
    print("⚠️ Google Drive への同期はスキップまたは失敗しました。")
    print("   ※スクリーニング結果はローカル超高速SSD（/content/working/output/）に正常に生成されています。")
    print("   Step 4 でのプレビューおよび Step 5 での直接ダウンロードはそのままご利用いただけます。")

# 調査用サマリを保存 (同期結果も記録)
diag_extra["Drive同期"] = "成功" if sync_ok else "スキップまたは失敗"
write_run_summary(
    diag_path, context=context, working_dir=Path("/content/working"), drive_dir=DRIVE_DIR,
    elapsed_sec=elapsed, extra=diag_extra, source=SOURCE_COMPLETED,
)


## 【Step 4】結果プレビュー & サマリ表示
ローカル作業層（`/content/working/output/`）の CSV を読み込み、スクリーニング適格銘柄のランキング Top 10（Grade S / A / B）および除外理由の内訳を表示します。
※本ツールは財務・市場指標に基づく客観的な機械スクリーニングを行うものであり、特定の有価証券の売買を推奨または助言するものではありません。

In [ ]:
from pathlib import Path

import polars as pl

daily_report_path = Path('/content/working/output/daily_report.csv')
uncalculable_path = Path('/content/working/output/uncalculable_stocks.csv')

if daily_report_path.exists():
    df_report = pl.read_csv(daily_report_path, null_values=['-', 'None', 'null', ''])
    if len(df_report) == 0:
        print('ℹ️ daily_report.csv は生成されましたが、適格銘柄は 0 社でした。足切り条件を緩和して再実行してください。')
    else:
        # 1. 格付け表記の統一 (万一旧キャッシュCSVが残っていても即座に Grade S/A/B/C へ変換)
        GRADE_MAP = {
            "STRONG_BUY": "Grade S",
            "BUY": "Grade A",
            "WATCH": "Grade B",
            "PASS": "Grade C",
        }
        if 'Verdict' in df_report.columns:
            df_report = df_report.with_columns(
                pl.col('Verdict').replace(GRADE_MAP).alias('Verdict')
            )
        
        # 表示用カラム (Verdict列を Grade 列名として直観的にプレビュー)
        display_df = df_report.rename({'Verdict': 'Grade'}) if 'Verdict' in df_report.columns else df_report
        display_cols = [c for c in ['Rank', 'Code', 'Name', 'Market', 'Grade', 'Score', 'PER', 'PBR', 'ROE', 'Equity_Ratio', 'Div_Yield'] if c in display_df.columns]
        # 【重要】上位候補は判定（Grade A以上 / BUY以上）で絞り込んで並べる
        grade_col = "Grade" if "Grade" in display_df.columns else ("Verdict" if "Verdict" in df_report.columns else None)
        if grade_col:
            qualified_top = display_df.filter(pl.col(grade_col).is_in(["Grade S", "Grade A", "STRONG_BUY", "BUY"]))
            if len(qualified_top) > 0:
                top10 = qualified_top.head(10).select(display_cols)
                print("=======================================================")
                print(f" 🏆 スクリーニング上位適格銘柄 Top 10 (Grade A以上: 全 {len(qualified_top)} 社)")
                print("=======================================================")
            else:
                top10 = display_df.head(10).select(display_cols)
                print("=======================================================")
                print(f" 🏆 スクリーニング適格銘柄ランキング Top 10 (全 {len(df_report)} 社)")
                print("=======================================================")
        else:
            top10 = display_df.head(10).select(display_cols)
            print("=======================================================")
            print(f" 🏆 スクリーニング適格銘柄ランキング Top 10 (全 {len(df_report)} 社)")
            print("=======================================================")
        
        
        try:
            from IPython.display import display
            display(top10.to_pandas())
        except Exception:
            print(top10)
            
        grade_col = 'Grade' if 'Grade' in display_df.columns else ('Verdict' if 'Verdict' in df_report.columns else None)
        if grade_col:
            grade_order = ['Grade S', 'Grade A', 'Grade B', 'Grade C']
            grade_counts = display_df.group_by(grade_col).len()
            counts_dict = {r[grade_col]: r['len'] for r in grade_counts.to_dicts()}
            print("\n📈 客観的格付けグレード (Grade) 別内訳:")
            for g in grade_order:
                if g in counts_dict:
                    print(f'   ・{g}: {counts_dict[g]} 社')
else:
    print('⚠️ daily_report.csv が見つかりません。Step 3 を実行してください。')

# 除外銘柄の理由別集計
if uncalculable_path.exists():
    df_rej = pl.read_csv(uncalculable_path, null_values=['-', 'None', 'null', ''])
    if 'filter_reason' in df_rej.columns:
        reason_counts = df_rej.group_by('filter_reason').len().sort('len', descending=True)
        print(f"\n📊 第1層 足切り除外銘柄の内訳 (全 {len(df_rej)} 社):")
        for row in reason_counts.to_dicts():
            r_name = row['filter_reason']
            r_len = row['len']
            print(f'   ・{r_name}: {r_len} 社')


## 【Step 5】CSV 保存確認 & ローカルダウンロード
生成された `daily_report.csv` を手元の PC へ直接ダウンロードします。  
- ※ブラウザ（Chrome 等）の仕様により、ダウンロードバーではなく**画面右上のトレイ（📥 アイコン）や「ダウンロード」フォルダに直接保存**されます。
- ※ブラウザの「ポップアップブロック」によりダウンロードが静かに遮断される場合があります。その場合はアドレスバー右端で許可するか、Google Drive / 画面左端 📁 から直接ファイルを取得してください。


In [ ]:
import os
from pathlib import Path

report_file = "/content/working/output/daily_report.csv"
drive_dir_display = os.environ.get("STOCK_ANALYZER_DRIVE_DIR", "MyDrive/StockAnalyzer")

if Path(report_file).exists():
    try:
        from google.colab import files
        print("📥 daily_report.csv のダウンロードをブラウザにリクエストしました。")
        files.download(report_file)
        print("✅ ダウンロード命令をブラウザへ送信しました！")
        print("   📂 お使いの PC の「ダウンロード」フォルダをご確認ください（Chrome等では右上の 📥 アイコンに通知されます）。")
        print("   ⚠️ 【もし保存されない場合】ブラウザのアドレスバー右端でポップアップがブロックされていないかご確認ください。")
        print(f"   ℹ️ なお、Google Drive（{drive_dir_display}/output/daily_report.csv）にも安全に保存されています。")
        print("   💡 画面左端の 📁（ファイル）➔ working ➔ output ➔ daily_report.csv を右クリックして手動保存も可能です。\n")
    except ImportError:
        print(f"ℹ️ ローカル環境のためブラウザダウンロードをスキップします。ファイルパス: {report_file}")
    except Exception as e:
        print(f"⚠️ ブラウザダウンロード要求中にエラーが発生しました: {e}")
        print(f"   ℹ️ Google Drive（{drive_dir_display}/output/daily_report.csv）には安全に保存されていますので、Driveから直接ご確認ください。")
        print("   💡 または、画面左端の 📁（ファイル）➔ working ➔ output ➔ daily_report.csv を右クリックしてダウンロードすることも可能です。\n")
else:
    print(f"⚠️ ファイルが見つかりません: {report_file}")


## 【困ったときは】調査用サマリの表示・保存
エラーが出た・結果がおかしいときだけ、このセルを実行してください（通常は不要です）。
実行環境・取得状況・エラー内容をまとめたテキストが表示され、ファイルとしてもダウンロードされます。
**APIキー・メールアドレス・認証情報は含まれません。** ご連絡の際に、表示内容またはダウンロードしたファイルを添えてください。


In [ ]:
from pathlib import Path

summary_path = Path("/content/working/output/diagnostics_summary.txt")

# Step 3 より前に問題が起きた場合など、サマリが未作成なら、その時点で収集できる情報から作成する
if not summary_path.exists():
    try:
        from src.utils.diagnostics import write_run_summary
        write_run_summary(
            summary_path,
            context=globals().get("context"),
            working_dir=Path("/content/working"),
            drive_dir=globals().get("DRIVE_DIR"),
            extra={
                "取得コード指定(TARGET_BRANCH)": globals().get("TARGET_BRANCH", "?"),
                "初回実行判定(現在の値)": globals().get("is_first_run", "?"),
                "reset_database": globals().get("reset_database", "?"),
            },
        )
    except Exception as e:
        print(f"⚠️ 調査用サマリを作成できませんでした ({type(e).__name__}: {e})。")
        print("   表示されているエラー画面のスクリーンショットを添えてご連絡ください。")

if summary_path.exists():
    print(summary_path.read_text(encoding="utf-8"))
    try:
        from google.colab import files
        files.download(str(summary_path))
    except Exception:
        print(f"ℹ️ ファイルの場所: {summary_path}")

---

### 📄 Stock Analyzer Core
- **Repository**: [https://github.com/akkyey/stock-analyzer-core](https://github.com/akkyey/stock-analyzer-core)
- **Author**: akkyey
- **License**: [MIT License](https://opensource.org/licenses/MIT) - Copyright (c) 2026 akkyey
